ARCHITETTURA BASE DELLE RETI RICORRENTI (RNN, LSTM, GRU)

- Il concetto di stato nascoso (hidden state), concetto fondamentale di queste reti
- La tecnica dello srotolamento (unrolling) per visualizzare il flusso temporale. tecmnica di visualizzazione
- La meccanica del calcolo manuale dello stato nascosto al primo passo temporale.

Come fa una rete a non dimenticare?

Lo Stato Nascosto: La Memoria della Rete
Codificare il passato per comprendere il presente.
Nelle architetture feed-forward, i dati fluiscono esclusivamente dall'input all'output senza memoria dei campioni passati. I dati entrano, vengono trasformati ed escono. Ogni campione è un isola felice senza passato
Invece
Le RNN rompono questo schema introducendo lo stato nascosto.
Lo stato nascosto agisce come una variabile di stato che viene aggiornata a ogni istante temporale, portando con se una traccia compressa di tutte le informazioni incontrate precedentemente nella sequenza.
In una RNN questo diario è un vettore che scorre all'interno della rete aggiornando i pesi costantemente

Pilastri dello Stato Nascosto
- Persistenza informativa: a differenza dei layer densi, lo stato nascosto non viene azzerato tra un elemento e l'altro della stessa sequenza. L'informazione non muore alla fine del calcolo ma sopravvive per il calcolo sucessivo.
- Dipendenza ricorsiva: il valore dello stato corrente è funzione matematica diretta dello stato calcolato nell'istante precedente. Ogni pensiero attuale è figlio del pensiero precedente.
- Vettore di contesto: lo stato nascosto può essere interpretato come un riassunto numerico del contesto temporale osservato fino a quel momento. Riassunto numerico di tutto ciò che la rete ha visto finora, questo è lo stato nascosto.
- Aggiornamento dinamico: la rete  utilizza metrici di pesi specifiche per integrare il nuovo input con la memoria esistente. La rete decide, tramite i pesi, quanto del presente integrare nella memoria esistente.

Dinamiche della Memoria Interna.
- Inizializzazione dello stato
All'inizio di ogni sequenza, lo stato nascosto deve essere inizializzato, solitamente con un vettore di zeri, rappresentando l'assenza di memoria pregressa. Ogni diario inizia con una pagina bianca
- Dimensione dello stato
Il numero di unità nel layer corrente definisce la 'larghezza' della memoria; uno stato più grande può teoricamente memorizzare pattern più complessi. E' la capacità della nostra memoria, un diario iù grande richiede anche risorse più importanti per essere gestito.
- Funzione di attivazione
Tradizionalmente si utilizza la tangente iperbolica per mantenere i valori dello stato confinati in un range controllato.

Confronto reti riccorrenti e reti statiche
Dalla trasformazione spaziale alla computazione temporale.
Mentre una CNN estrae feature spaziali tramite filtri fissi, una RNN estrae feature temporali modificando il proprio stato interno. Una RNN analizza il tempo. Questa 'ricorsione' permette di gestire input di lunghezza variabile.
Senza lo stato nascosto, la rete non avrebbe modo di distringuere se un elemento appare all'inizio, a metà o alla fine di una frase o di una serie storica.

Per addestrare questi reti, abbiamo bisogno di una prospettiva diversa, dobbiamo srotolarle.

Unrolling: Srotolare il Tempo
Visualizzare la ricorsione come una catena di operazioni
Il concetto di 'unrolling' o srotolamento è fondamentale per comprendere come avviene l'addestramento. Consiste nel rappresentare la rete ricorrente come una sequenza di copie identiche.
Ogni 'copia' della rete corrisponde a un singolo passo temporale, dove l'output dello stato nascosto di una copia diventa l'input per la successiva.

Meccanismi dello Srotolamento
Dal ciclo logico alla struttura sequenziale.
Sembre una rete molto profonda, ma c'è una differenza fondamentale, la condivisione dei pesi.
- Condivisione dei pesi: le matrici di peso sono identiche per ogni passo temporale srotolato, garantendo inviarianza temporale. le matrici dei pesi che vedo in ogni cella sono le stesse, questo non solo risparmia la memoria, ma permette alla rete di imparare pattern che non dipendono dal momento in cui accade 
- Flusso di gradiente: lo srotolamento trasforma la ricorsione in un grafo aciclico diretto, permettendo l'uso della backpropagation
- Profondità temporale: una sequenza di lunghezza T srotolata equivale a  una rete profonda T layer, con sfide specifiche per l'ottimizzazione
- Visualizzazione del grafo: ogni cella riceve in input 'x' e lo stato 'h' dal passato, emettendo un nuovo 'h' per il futuro

Ma se srotoliamo tutto, come facciamo ad insegnare alla rete come imparare?

Implicazione dello Srotolamento
Backpropagation Througn Time (BPTT): l'algoritmo di addestramento che calcola i gradienti propagando l'errore a ritroso attraverso tutti i passi temporali srotolati
Invarianza dei parametri: poichè i pesi sono condivisi, la rete impara pattern che non dipendono dalla posizione assoluta nella sequenza.
Costo computazionale: lo srotolamento richiede la memorizzazione di tutti gli stati intermedi per il calcolo dei gradienti, aumentando l'uso della memoria.
Più la sequanza è lunga, più il viaggio nel tempo è complesso.

Limiti teorici dello srotolamento 
Il problema della memoria a lungo termine
Srotolare una rete per sequenza molto lunghe porta a grafi estremamente profondi. Questo introduce problemi di stabilità numerica durante il calcolo dei gradienti, noti come vanisching gradient.
Il segnale dell'errore tende a svanire man mano che cerchiamo di portarlo verso l'inizio del tempo.
Per mitigare questo, ed evitare che la memoria GPU esploda, si utilizzano tecniche come il 'truncated BPTT', dove lo srotolamento viene limitato a un numero fisso di passi temporali, accettando un compromesso tra memoria e capacità di apprendimento.



In [1]:
import numpy as np
import tensorflow as tf

# =================================================================
# 1. IMPLEMENTAZIONE MANUALE CON NUMPY
# =================================================================

def rnn_cell_forward(xt, h_prev, Wxh, Whh, bh):
    """
    Calcola lo stato nascosto corrente h_t basandosi sull'input x_t 
    e lo stato precedente h_{t-1}.
    """
    # Calcolo della pre-attivazione (combinazione lineare)
    z_t = np.dot(Wxh, xt) + np.dot(Whh, h_prev) + bh
    
    # Applicazione della funzione di attivazione non lineare (tanh)
    h_t = np.tanh(z_t)
    
    return h_t

# Definizione dei parametri (esempio giocattolo)
dim_input = 1    # x_t è un singolo valore (es. temperatura)
dim_hidden = 2   # lo stato nascosto ha 2 neuroni

# Inizializzazione pesi casuali
Wxh = np.array([[0.5], [0.8]])   # Forma (m, d) -> (2, 1)
Whh = np.array([[0.1, 0.2],      # Forma (m, m) -> (2, 2)
                [0.3, 0.4]])
bh = np.zeros((dim_hidden, 1))

# Dati di input
h0 = np.zeros((dim_hidden, 1))   # Stato iniziale (tabula rasa)
x1 = np.array([[1.0]])           # Primo elemento della sequenza

# Calcolo manuale di h1
h1 = rnn_cell_forward(x1, h0, Wxh, Whh, bh)

print("--- Calcolo Manuale ---")
print(f"Stato nascosto h1:\n{h1}")

# =================================================================
# 2. VERIFICA CON KERAS (SimpleRNN)
# =================================================================

# Prepariamo l'input per Keras: (batch, timesteps, features)
keras_input = np.array([[[1.0]]]) 

rnn_layer = tf.keras.layers.SimpleRNN(units=dim_hidden, activation='tanh', return_state=True)

# Per fini didattici, forziamo i pesi di Keras a essere uguali ai nostri
# Keras usa (input_dim, units) per Wxh e (units, units) per Whh
rnn_layer.build((None, 1, 1))
rnn_layer.set_weights([Wxh.T, Whh.T, bh.flatten()])

output, h1_keras = rnn_layer(keras_input)

print("\n--- Verifica con Keras ---")
print(f"Stato nascosto h1 (Keras):\n{h1_keras.numpy().T}")

--- Calcolo Manuale ---
Stato nascosto h1:
[[0.46211716]
 [0.66403677]]

--- Verifica con Keras ---
Stato nascosto h1 (Keras):
[[0.46211714]
 [0.6640368 ]]


Lo strato nascosto è la memoria delle RNN.
Srotolare la rete facilita la visualizzazione del flusso dati e abilita l'addestramento tramite backpropagation throut time
L'aggiornamento dello stato avviene tramite combinazioni lineari di input e stato precedente, mediate da una funzione di attivazione come tanh